In [14]:
import pandas as pd
import numpy as np

cards_df = pd.read_csv('./data/cards_clustered.csv')
decks_df = pd.read_csv('./data/decks.csv')

In [15]:
def build_cluster_deck_completer(user_input_cards, cards_df, decks_df):
    """
    Completes a partial deck by matching its cluster ratio blueprint to the 
    closest top-tier meta deck, breaking ties with Win Rate, and filling 
    slots with the highest win rate cards from the missing clusters.
    """
    # -------------------------------------------------------------
    # Setup & Preprocessing
    # -------------------------------------------------------------
    card_cols = [f'card{i}' for i in range(1, 9)]
    
    # 1. Create mapping dictionaries from cards_df
    card_to_cluster = dict(zip(cards_df['name'], cards_df['Cluster']))
    card_to_wr = dict(zip(cards_df['name'], cards_df['Win Rate']))
    
    # Get a unique list of all clusters present in your model (ordered 0, 1, 2...)
    unique_clusters = sorted(cards_df['Cluster'].unique())
    cluster_columns = [f"Cluster_{int(c)}" for c in unique_clusters]
    
    # 2. Convert the master decks database into structural cluster counts
    deck_clusters = decks_df[card_cols].map(lambda x: card_to_cluster.get(x, np.nan))
    
    # Pre-calculate counts of each cluster for all database decks
    db_counts = deck_clusters.apply(lambda row: row.value_counts(), axis=1).fillna(0).astype(int)
    db_counts = db_counts.reindex(columns=unique_clusters, fill_value=0)
    db_counts.columns = cluster_columns
    
    # Convert counts to ratios (each deck has exactly 8 cards)
    db_ratios = db_counts.div(8)
    
    # -------------------------------------------------------------
    # Step 1 & 2: Process User Input & Convert to Cluster Ratios
    # -------------------------------------------------------------
    # Convert input list to its corresponding cluster counts
    input_clusters = [card_to_cluster.get(card) for card in user_input_cards if card in card_to_cluster]
    
    input_counts_series = pd.Series(input_clusters).value_counts()
    input_counts = input_counts_series.reindex(unique_clusters, fill_value=0)
    
    # Important: Normalize by 8 because we are comparing to full-deck ratios
    input_ratio = (input_counts / 8).values 
    
    # -------------------------------------------------------------
    # Step 3: Compute Distances (Manual KNN to handle tie-breakers)
    # -------------------------------------------------------------
    # Calculate Euclidean distance from the input ratio to every deck in the database
    distances = np.linalg.norm(db_ratios.values - input_ratio, axis=1)
    
    # Create a temporary evaluation dataframe
    eval_df = decks_df.copy()
    eval_df['distance'] = distances
    
    # Sort primarily by closest distance (ascending), and secondarily by Win Rate (descending)
    eval_df = eval_df.sort_values(by=['distance', 'Win Rate'], ascending=[True, False]).reset_index(drop=True)
    
    # Grab the absolute closest structural match (breaking ties with highest Win Rate)
    target_deck = eval_df.iloc[0]
    target_deck_idx = eval_df.index[0]
    
    # Extract the target deck's cluster count blueprint
    target_blueprint = db_counts.iloc[target_deck_idx].to_dict()
    
    # -------------------------------------------------------------
    # Step 4: Locate Needed Clusters & Fill Slots Safely
    # -------------------------------------------------------------
    # Determine which cluster categories are still missing compared to the target blueprint
    current_counts = input_counts_series.reindex(unique_clusters, fill_value=0).to_dict()
    
    needed_clusters = []
    for c in unique_clusters:
        c_col = f"Cluster_{int(c)}"
        slots_needed = target_blueprint[c_col] - current_counts.get(c, 0)
        if slots_needed > 0:
            needed_clusters.extend([c] * slots_needed)
            
    # Track cards we cannot use (no duplicates allowed from input)
    used_cards = set(user_input_cards)
    recommended_fills = []
    
    # Fill each missing structural slot sequentially
    for cluster_needed in needed_clusters:
        # Filter pool: Cards in this specific cluster, not already in the deck
        available_pool = cards_df[(cards_df['Cluster'] == cluster_needed) & (~cards_df['name'].isin(used_cards))]
        
        if not available_pool.empty:
            # Pick the top individual card in that cluster sorted by its standalone Win Rate
            best_card = available_pool.sort_values(by='Win Rate', ascending=False).iloc[0]['name']
            recommended_fills.append(best_card)
            used_cards.add(best_card) # Add to set to prevent future duplicate selections
        else:
            # Fallback guardrail: If a small cluster is completely exhausted, grab any top tier card left
            fallback_pool = cards_df[~cards_df['name'].isin(used_cards)]
            best_card = fallback_pool.sort_values(by='Win Rate', ascending=False).iloc[0]['name']
            recommended_fills.append(best_card)
            used_cards.add(best_card)

    # Output details
    print(f"--- Deck Builder Result ---")
    print(f"Matched Archetype Blueprint: {target_deck['card1']}, {target_deck['card2']}, ... (Base WR: {target_deck['Win Rate']}%)\n")
    print(f"User Input:        {user_input_cards}")
    print(f"Suggested Fills:   {recommended_fills}")
    print(f"Final 8-Card Deck: {user_input_cards + recommended_fills}")
    
    return user_input_cards + recommended_fills

# -------------------------------------------------------------
# Example Execution Call:
# -------------------------------------------------------------
# final_deck = build_cluster_deck_completer(['Lumberjack', 'Balloon'], cards_df, decks_df)

In [16]:
build_cluster_deck_completer(['Hog Rider', 'Cannon', 'Ice Spirit', 'Musketeer', 'Fireball', 'Ice Golem'], cards_df, decks_df)

--- Deck Builder Result ---
Matched Archetype Blueprint: Skeleton Barrel, Bowler, ... (Base WR: 63.2%)

User Input:        ['Hog Rider', 'Cannon', 'Ice Spirit', 'Musketeer', 'Fireball', 'Ice Golem']
Suggested Fills:   ['Cannon Cart', 'Balloon', 'Sparky', 'Barbarian Barrel']
Final 8-Card Deck: ['Hog Rider', 'Cannon', 'Ice Spirit', 'Musketeer', 'Fireball', 'Ice Golem', 'Cannon Cart', 'Balloon', 'Sparky', 'Barbarian Barrel']


['Hog Rider',
 'Cannon',
 'Ice Spirit',
 'Musketeer',
 'Fireball',
 'Ice Golem',
 'Cannon Cart',
 'Balloon',
 'Sparky',
 'Barbarian Barrel']